[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filipepacheco/music-theory-lab/blob/claude/init-project-mdDKh/research/chord-recognition/chord_recognition.ipynb)

# Chord Recognition Research

Run this notebook on **Google Colab** (Runtime → Run all, or Ctrl+F9).

What this does:
1. Installs and patches madmom (handles Python 3.10+ incompatibilities automatically)
2. Lets you upload an mp3
3. Runs madmom's CNN + CRF chord recognizer on it
4. Prints timestamped chord segments
5. Runs autochord (richer vocabulary — recognises 7th chords) and compares

**Ground truth comparison**: if you know the chords of the song you upload, note them down before running — comparing what you hear vs what the models say is the main learning here.

In [ ]:
# Install dependencies
# madmom needs numpy + cython available at build time, hence the order
import sys

!{sys.executable} -m pip install -q 'numpy>=1.26' cython
!{sys.executable} -m pip install -q madmom==0.16.1 --no-build-isolation
!{sys.executable} -m pip install -q librosa mir_eval soundfile

print('Install complete')

In [ ]:
# Patch madmom 0.16.1 for Python 3.10+ and numpy 1.24+
# madmom uses collections.MutableSequence (removed in Py3.10) and
# np.float / np.int / np.bool / np.object (removed in numpy 1.24)
import re
import sysconfig
from pathlib import Path

NUMPY_ALIASES = [
    (re.compile(r'\bnp\.float\b(?!\d)'), 'float'),
    (re.compile(r'\bnp\.int\b(?!\d|p\b)'), 'int'),
    (re.compile(r'\bnp\.bool\b(?!_)'), 'bool'),
    (re.compile(r'\bnp\.object\b(?!_)'), 'object'),
]
COLLECTIONS_FIX = (
    re.compile(r'^from collections import MutableSequence\b', re.MULTILINE),
    'from collections.abc import MutableSequence',
)

madmom_root = Path(sysconfig.get_paths()['purelib']) / 'madmom'
changed = []
for py_file in madmom_root.rglob('*.py'):
    text = py_file.read_text()
    patched = text
    for pattern, replacement in NUMPY_ALIASES + [COLLECTIONS_FIX]:
        patched = pattern.sub(replacement, patched)
    if patched != text:
        py_file.write_text(patched)
        changed.append(py_file.name)

print(f'Patched {len(changed)} file(s): {changed}')

In [ ]:
# Upload your mp3 (or wav)
from google.colab import files

uploaded = files.upload()  # opens file picker
audio_path = next(iter(uploaded.keys()))
print(f'Using: {audio_path}')

In [ ]:
# Run chord recognition
# NOTE: madmom uses a 'majmin' vocabulary — it only outputs major and minor chords.
# Dominant 7ths (E7, A7, B7) map to their major root (E:maj, A:maj, B:maj).
# See the autochord cell below for a model with a richer chord vocabulary.
from madmom.features.chords import CNNChordFeatureProcessor, CRFChordRecognitionProcessor

print(f'Analyzing {audio_path} ...')
feat = CNNChordFeatureProcessor()(audio_path)
segments = CRFChordRecognitionProcessor()(feat)

results = [(float(s), float(e), str(label)) for s, e, label in segments]
print(f'Done. {len(results)} segments detected.\n')

print(f'{"Start":>8}  {"End":>8}  Label')
print('-' * 30)
for start, end, label in results:
    duration = end - start
    print(f'{start:8.2f}  {end:8.2f}  {label}  ({duration:.2f}s)')

In [ ]:
# Summary statistics
from collections import Counter

total_duration = results[-1][1] if results else 0
label_durations = Counter()
for start, end, label in results:
    label_durations[label] += end - start

print(f'Total duration: {total_duration:.1f}s')
print(f'Unique labels: {len(label_durations)}')
print()
print('Label coverage (sorted by duration):')
for label, dur in sorted(label_durations.items(), key=lambda x: -x[1]):
    pct = 100 * dur / total_duration if total_duration else 0
    print(f'  {label:<12}  {dur:5.1f}s  ({pct:.1f}%)')

In [ ]:
# Alternative: autochord — richer vocabulary (7, maj7, min7, sus4, ...)
# madmom's CRF is trained on majmin only; autochord uses a separate model
# that can distinguish dominant 7ths from plain major chords.
import sys
!{sys.executable} -m pip install -q autochord

import autochord

print(f'Analyzing {audio_path} with autochord ...')
raw = autochord.recognize(audio_path)
results_auto = [(float(s), float(e), str(lab)) for s, e, lab in raw]

print(f'Done. {len(results_auto)} segments.\n')
print(f'{"Start":>8}  {"End":>8}  Label')
print('-' * 30)
for start, end, label in results_auto:
    print(f'{start:8.2f}  {end:8.2f}  {label}  ({end - start:.2f}s)')

# Side-by-side vocabulary comparison
madmom_labels = sorted({lab for _, _, lab in results if lab != 'N'})
auto_labels   = sorted({lab for _, _, lab in results_auto if lab != 'N'})
print(f'\nmadmom unique labels : {madmom_labels}')
print(f'autochord unique labels: {auto_labels}')

In [ ]:
# Export results as a .chords.txt file (mir_eval-compatible, tab-separated)
# Download it to compare against ground truth later
from pathlib import Path
from google.colab import files as colab_files

stem = Path(audio_path).stem
out_name = f'{stem}.chords.txt'
with open(out_name, 'w') as f:
    for start, end, label in results:
        f.write(f'{start:.3f}\t{end:.3f}\t{label}\n')

colab_files.download(out_name)
print(f'Downloaded {out_name}')

## What to look for

- **`N`** (no chord) — model has low confidence. Heavy `N` coverage suggests the model struggles with this song's timbre or density.
- **Root accuracy** — does the letter name match even when the quality (maj/min/7) is wrong?
- **Boundary accuracy** — do chord changes happen around the right timestamps?
- **Failure patterns** — does it fail on specific sections (intro, chorus, bridge)? On specific chord types?

## Vocabulary limitation

madmom's CRF uses a **`majmin` vocabulary** (24 labels: 12 roots × maj/min + `N`). Dominant 7ths, major 7ths, minor 7ths, and suspensions all collapse to their closest major root. This is a model constraint, not a bug.

The **autochord** cell above uses a different pre-trained model with a fuller vocabulary — it should label E7 as `E:7` instead of `E:maj`. Compare the two outputs: roots should mostly agree; chord quality should be more accurate in autochord for 7th-heavy music (blues, jazz, funk).

For state-of-the-art accuracy (7ths, extensions, suspensions), the next step is the **BTC model** (Bi-directional Transformer for Chord Recognition), which tops MIREX benchmarks for full-vocabulary chord estimation.

## Chord label format (MIREX)

- `C:maj`, `C:min` — C major / minor
- `C:7`, `C:maj7`, `C:min7` — dominant, major, and minor sevenths
- `N` — no chord / silence

## Next steps

After running 3–5 songs, log patterns in `FINDINGS.md` in the repo:
- What does madmom get right vs. autochord?
- Where do both models fail?
- Does genre, tempo, or production style seem to matter?

That shapes the next experiment: BTC model comparison, source separation (Demucs) as preprocessing, or formal benchmark evaluation on Isophonics/McGill Billboard datasets with `mir_eval`.